In [1]:
# Imports

#Pytorch
import torch
import torch.nn as nn
from torch.nn.parameter import Parameter
import torch.utils.data as data

#Others
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_percentage_error,
    mean_squared_error,
    root_mean_squared_error,
    mean_absolute_error,
    r2_score,
    confusion_matrix
)
from sklearn.model_selection import train_test_split

from abc import abstractmethod

In [4]:
from ucimlrepo import fetch_ucirepo

# Functions

In [5]:
def generalized_bell(x, p):
    '''
    Calculates Generalized Bell values based on the input data and premise parameters of the ANFIS model.

    Parameters:
    - x (torch.Tensor): Input data tensor (as a batch or a single input).
    - p (torch.Tensor): 3D tensor containing 'A', 'B' and 'C' parameters by data dimension
                        and by ANFIS rule. The parameters represents the shape of the Bell curve,
                        meaning the width, slope and height respectively.

    Returns:
    - torch.Tensor: 2D or 3D tensor with Resulting Generalized Bell values (for single or batch input
                    respectively).

    Explanation:
    This function calculates Generalized Bell values based on the input data `x` and the parameters `p`.
    It ensures that the standard deviation values in `p` are not zero by replacing them with a small
    value (1e-6). The formula used to compute the Generalized Bell values is:

    \[ \frac{1}{1 + \exp\left(-0.5 \cdot \left(\frac{|x - p[:, :, 2}|}{p[:, :, 0]}\right)^2\right)} \]

    Note:
    - If `p[:, :, 0]` is zero, it is replaced with 1e-6 to prevent division by zero.

    '''
    return 1/(1 + torch.pow(torch.abs((x - p[:, :, 2])/torch.where(p[:, :, 0] == 0, torch.tensor(1e-6), p[:, :, 0])), 2*p[:, :, 1]))

<>:2: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_3090/4188964859.py:2: SyntaxWarning: invalid escape sequence '\['
  '''


In [6]:
def gaussian2(x, p):
    '''
    Calculates Gaussian values based on the input data and premise parameters of the ANFIS model 
    (is used as a membership function). It also works with more than one input (batches).

    Parameters:
    - x (torch.Tensor): Input data tensor (as a batch or a single input).
    - p (torch.Tensor): 3D tensor containing 'mu' and 'sigma' parameters by data dimension
                        and by ANFIS rule.

    Returns:
    - torch.Tensor: Tensor with resulting Gaussian values (either as a batch or a single output).

    Explanation:
    This function calculates Gaussian values based on the input data `x` and the parameters `p`.
    It ensures that the standard deviation values in `p` are not zero by replacing them with a small
    value (1e-6). The formula used to compute the Gaussian values is:

    \[ \exp\left(-0.5 \cdot \left(\frac{x - p[:, :, 0]}{p[:, :, 1]}\right)^2\right) \]

    Note:
    - If `p[:, :, 1]` is zero, it is replaced with 1e-6 to prevent division by zero.

    '''
    return torch.exp(-0.5 * torch.pow((x - p[:, :, 0])/torch.where(p[:, :, 1] == 0, torch.tensor(1e-6), p[:, :, 1]), 2))

<>:2: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_3090/4237469877.py:2: SyntaxWarning: invalid escape sequence '\['
  '''


In [7]:
def weighted_linear(x, c, w):
    """
    Compute the weighted linear combination of input features and coefficients (representing the
    consequent parameters of the ANFIS model). It is manufactured in such a way that it also works with
    more than one input (batches) and will be used to calculate the individual outputs of each rule of
    the ANFIS model.

    Parameters:
    - x (torch.Tensor): Input data tensor.
    - c (torch.Tensor): Coefficients tensor for the linear combination (consequent parameters).
    - w (torch.Tensor): Weights tensor for element-wise multiplication.

    Returns:
    - torch.Tensor: Resulting weighted linear combination.

    Explanation:
    This function calculates the weighted linear combination of the input features `x` and the coefficients `c`.
    It involves matrix multiplication of the input features by the transposed coefficients (excluding the last column),
    adding the last column of coefficients, and then element-wise multiplication by the weights `w`.
    The formula used is:

    \[ (x \cdot c[:, :-1].T + c[:, -1]) \cdot w \]

    Note:
    - The weights `w` are applied element-wise.

    """
    return (torch.bmm(x.unsqueeze(0).expand(c[:, :, :-1].size(0), -1, -1), torch.transpose(c[:, :, :-1], 1, 2)) + c[:, :, -1].unsqueeze(1)).mul(w.unsqueeze(0))

<>:2: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_3090/724767595.py:2: SyntaxWarning: invalid escape sequence '\['
  """


# ANFIS

## Layer 1: Fuzzify

In [8]:
class FuzzifyLayer(nn.Module):
    
    def __init__(self, x_train, init_fuzzy_rules=1, mf=generalized_bell, mf_params=['a', 'b', 'c'], init_mode=0):
        super(FuzzifyLayer, self).__init__()
        # Input data info
        self.input_size = x_train.shape[1]
        self.dtype = x_train.dtype
        self.max_val = torch.max(x_train).round().item()
        self.min_val = torch.min(x_train).round().item()
        
        self.input_shape = x_train.shape # QUIZAS SEA MAS CONVENIENTE OBTENER INFO DIRECTAMENTE DESDE SHAPE
        
        # Fuzzification parameters
        self.mf = mf
        self.mf_params = mf_params

        # Initialize premises
        self.premises = Parameter(self.init_premises(x_train=x_train, init_fuzzy_rules=init_fuzzy_rules, init_mode=init_mode), requires_grad=True)



    def init_premises(self, x_train, init_fuzzy_rules, init_mode):
        premises = torch.zeros(self.input_size, init_fuzzy_rules, len(self.mf_params), dtype=self.dtype)
        
        if init_mode != 0:
            premises += 2 * torch.rand(self.input_size, init_fuzzy_rules, len(self.mf_params), dtype=self.dtype) - 1

        elif self.mf == gaussian2:
            if init_fuzzy_rules > 1:
                min = torch.min(x_train, dim=0).values
                max = torch.max(x_train, dim=0).values
                stp = (max - min) / (init_fuzzy_rules - 1)
                for i in range(self.input_size):
                    h = torch.arange(min[i], max[i] + stp[i], stp[i])
                    premises[i, :, 0] = h[:init_fuzzy_rules]
                    premises[i, :, 1] = stp[i]/2
            else:
                for i in range(self.input_size):
                    premises[i, :, 0] = torch.mean(x_train[:, i])
                    premises[i, :, 1] = torch.std(x_train[:, i])

        elif self.mf == generalized_bell:
            if init_fuzzy_rules > 1:
                min = torch.min(x_train, dim=0).values
                max = torch.max(x_train, dim=0).values
                stp = (max - min) / (init_fuzzy_rules - 1)
                for i in range(self.input_size):
                    h = torch.arange(min[i], max[i] + stp[i], stp[i])
                    premises[i, :, 2] = h[:init_fuzzy_rules] # C : centro
                    premises[i, :, 0] = stp[i]/2 # A : ancho
                    premises[i, :, 1] = 8 # B : pendiente
            else:
                for i in range(self.input_size):
                    premises[i, :, 2] = torch.mean(x_train[:, i]) # C : centro
                    premises[i, :, 0] = torch.std(x_train[:, i]) # A : ancho
                    premises[i, :, 1] = 2 # B : pendiente

        return premises



    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)



    @property
    def premises_structure(self):
        df = pd.DataFrame()
        rules = ['Fuzzy rule {}'.format(i) for i in range(1, self.premises.data.shape[1]+1)]
        
        for i in range(self.input_size):
            for j in range(len(self.mf_params)):
                column = pd.Series(self.premises.data[i,:,j], index=rules, name=self.mf_params[j] + f' (x{i})', )
                df[self.mf_params[j] + f' (x{i})'] = column

        return df


    
    @property
    def plot_premises(self):
        variables = [f'x{i}' for i in range(self.input_size)]
        dataframe = self.premises_structure
        
        sep = round((0.1*(self.max_val - self.min_val))) + 1
        x = np.linspace(self.min_val - sep, self.max_val + sep, 500) 

        fig, axes = plt.subplots(nrows=self.premises.data.shape[1], ncols=len(variables), figsize=(15, 8), sharex=True, sharey=True)

        for i, rule in enumerate(dataframe.index):
            for j, var in enumerate(variables):

                if self.mf == gaussian2:
                    mu = dataframe.loc[rule, f'mu ({var})']
                    sigma = dataframe.loc[rule, f'sigma ({var})']
                    y = np.exp(-((x - mu) ** 2) / (2 * sigma ** 2))

                elif self.mf == generalized_bell:
                    a = dataframe.loc[rule, f'a ({var})']
                    b = dataframe.loc[rule, f'b ({var})']
                    c = dataframe.loc[rule, f'c ({var})']
                    y = 1/(1 + np.power(np.abs((x - c) / a), 2*b))

                ax = axes[i, j]
                ax.plot(x, y, label=f'{rule}, {var}')
                ax.set_title(f'{rule}, {var}')
                ax.grid(True)
                if i == self.premises.data.shape[1] - 1:
                    ax.set_xlabel('x')
                if j == 0:
                    ax.set_ylabel('Membership Value')

        plt.tight_layout()
        plt.show()

        return None

## Layer 2: Firing Levels

In [9]:
class r_FiringLevelsLayer(nn.Module):
    
    def __init__(self):
        super(r_FiringLevelsLayer, self).__init__()


    def forward(self, m):
        w = m.prod(dim=m.dim()-2)
        return w

## Layer 3: Normalization

In [10]:
class NormalizationLayer(nn.Module):
    
    def __init__(self):
        super(NormalizationLayer, self).__init__()


    def forward(self, w):
        sum = torch.sum(w, dim=-1, keepdim=True)
        sum[sum == 0] = 1
        w = w/sum
        return w

## Layer 4: Consequent

In [11]:
class ConsequentLayer(nn.Module):
    
    def __init__(self, input_size, dtype, init_consequent_rules=1, function=weighted_linear, outputs=1):
        super(ConsequentLayer, self).__init__()
        self.function = function
        self.consequents = Parameter(2 * torch.rand(outputs, init_consequent_rules, input_size + 1, dtype=dtype) - 1, requires_grad=True)


    def forward(self, x, w):
        outputs = self.function(x, self.consequents, w)
        return outputs


    @property
    def consequents_structure(self):
        dfs = [pd.DataFrame() for _ in range(self.consequents.data.shape[0])]

        rules = ['rule {}'.format(i) for i in range(1, self.consequents.data.shape[1]+1)]

        for o in range(self.consequents.data.shape[0]):
            for i in range(self.consequents.data.shape[2]):
                name=f'c{i} (x{i})'
                if (i == self.consequents.data.shape[2]-1): name=f'c{i}'
                column = pd.Series(self.consequents.data[o,:,i], index=rules, name=name)
                dfs[o][name] = column

        return dfs

## Layer 5: Output

In [12]:
class OutputLayer(nn.Module):

    def __init__(self):
        super(OutputLayer, self).__init__()

    def forward(self, x):
        return torch.sum(x, dim=-1)

## Implementation

In [14]:
class rANFIS(nn.Module):

    def __init__(self, x_train, init_fuzzy_rules=1, cf=weighted_linear, mf=generalized_bell, mf_params=['a', 'b', 'c'], init_mode=0, output_type=None, outputs_amount=1):
        super(rANFIS, self).__init__()
        self.input_size = x_train.shape[1]
        self.dtype = x_train.dtype
        self.mf_params = mf_params

        self.fuzzify_layer = FuzzifyLayer(x_train, init_fuzzy_rules, mf, mf_params, init_mode)
        self.firing_levels_layer = r_FiringLevelsLayer()
        self.normalization_layer = NormalizationLayer()
        self.consequent_layer = ConsequentLayer(self.input_size, self.dtype, init_fuzzy_rules, cf, outputs_amount)
        self.output_layer = OutputLayer()

        if (output_type == None):
            self.last_layer = nn.Identity()
        elif (output_type == 'binary'):
            self.last_layer = nn.Sigmoid()
        elif (output_type == 'multiclass'):
            self.last_layer = nn.Softmax(dim=0)


    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.normalization_layer(self.firing_levels_layer(output)))
        output = self.output_layer(output)
        output = self.last_layer(output)
        return output


    def classes_prediction(self, x):
        with torch.no_grad():
            output = self.forward(x)

        if isinstance(self.last_layer, nn.Softmax):
            output = torch.max(output, dim=0).indices

        elif isinstance(self.last_layer, nn.Sigmoid):
            output = (output >= 0.5).to(int)

        return output


    def intermediate_values(self, x):
        with torch.no_grad():
          w = self.fuzzify_layer(x)
          w = self.firing_levels_layer(w)
          w_norm = self.normalization_layer(w)
          outputs = self.consequent_layer(x, w_norm)
        return w, w_norm, outputs


    @property
    def rules(self):
        return self.consequents.shape[1]


    @property
    def premises_structure(self):
        return self.fuzzify_layer.premises_structure
    
    @property
    def plot_premises(self):
        return self.fuzzify_layer.plot_premises


    @property
    def premises(self):
        return self.fuzzify_layer.premises.data


    def set_premises(self, premises):
        self.fuzzify_layer.premises = Parameter(premises, requires_grad=True)


    @property
    def consequents_structure(self):
        return self.consequent_layer.consequents_structure


    @property
    def consequents(self):
        return self.consequent_layer.consequents.data


    def set_consequents(self, consequents):
        self.consequent_layer.consequents = Parameter(consequents, requires_grad=True)

### Test

In [15]:
x_train = torch.randn(5, 2)
anfis = rANFIS(x_train, init_fuzzy_rules=3, cf=weighted_linear, mf=generalized_bell, mf_params=['a', 'b', 'c'], init_mode=0, output_type="binary", outputs_amount=2)
anfis.premises_structure

,a (x0),b (x0),c (x0),a (x1),b (x1),c (x1)
Fuzzy rule 1,0.186266,8.0,-0.290515,0.738891,8.0,-1.354418
Fuzzy rule 2,0.186266,8.0,0.082016,0.738891,8.0,0.123363
Fuzzy rule 3,0.186266,8.0,0.454547,0.738891,8.0,1.601145


In [16]:
consequents_dfs = anfis.consequents_structure

i = 1
for df in consequents_dfs:
    print(f"Output {i}")
    display(df)
    i+=1
    print("\n\n")

Output 1


,c0 (x0),c1 (x1),c2
rule 1,-0.122694,0.133893,0.354219
rule 2,-0.249422,-0.375102,-0.180894
rule 3,-0.279953,-0.421614,-0.767089





Output 2


,c0 (x0),c1 (x1),c2
rule 1,-0.586169,-0.357632,-0.316260
rule 2,0.013941,-0.692699,-0.472985
rule 3,-0.700075,-0.004227,0.199982


In [34]:
anfis = rANFIS(x_train, init_fuzzy_rules=3, cf=weighted_linear, mf=generalized_bell, mf_params=['a', 'b', 'c'], init_mode=0)
anfis(x_train).shape


torch.Size([1, 136])

In [35]:
anfis = rANFIS(x_train, init_fuzzy_rules=3, cf=weighted_linear, mf=generalized_bell, mf_params=['a', 'b', 'c'], init_mode=0, output_type="binary")
anfis(x_train).shape

torch.Size([1, 136])

In [36]:
anfis = rANFIS(x_train, init_fuzzy_rules=3, cf=weighted_linear, mf=generalized_bell, mf_params=['a', 'b', 'c'], init_mode=0, output_type="multiclass", outputs_amount=3)
anfis(x_train).shape

torch.Size([3, 136])

# Pre

## Early stopping

In [17]:
class EarlyStopping():
    def __init__(self, patience, delta=0, last_state=False):
        #Parameters
        self.patience = patience
        self.delta = delta
        self.last_state = last_state

        #For running
        self.counter = 0
        self.best_loss = None
        self.best_premises = None
        self.best_consequents = None
        self.early_stop = False

    def __call__(self, loss, ANFISmodel):
        if self.best_loss is None:
            self.best_loss = loss
            self.best_premises = ANFISmodel.premises
            self.best_consequents = ANFISmodel.consequents

        elif loss + self.delta > self.best_loss:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                if self.last_state == False:
                    ANFISmodel.set_premises(self.best_premises)
                    ANFISmodel.set_consequents(self.best_consequents)

        else:
            self.best_loss = loss
            self.best_premises = ANFISmodel.premises
            self.best_consequents = ANFISmodel.consequents
            self.counter = 0

    def reset(self):
        self.counter = 0
        self.best_loss = None
        self.best_premises = None
        self.best_consequents = None
        self.early_stop = False

## Measures

In [18]:
def obtain_measures(ANFISmodel, x, y):
    measures = {}
    with torch.no_grad():
        pred = ANFISmodel(x)

    if isinstance(ANFISmodel.last_layer, nn.Identity):
        #numpy conversion for sklearn functions
        pred = pred.numpy()
        y = y.numpy()

        mse = mean_squared_error(y, pred)
        measures['mse'] = mse

        rmse = root_mean_squared_error(y, pred)
        measures['rmse'] = rmse

        variance = np.var(y)
        nmse = mse / variance
        measures['nmse'] = nmse

        mape = mean_absolute_percentage_error(y, pred)
        measures['mape'] = mape

        mae = mean_absolute_error(y, pred)
        measures['mae'] = mae

        r2 = r2_score(y, pred)
        measures['R2'] = r2

    elif isinstance(ANFISmodel.last_layer, nn.Sigmoid):
        #pytorch calculation for bce calculation
        bce = nn.functional.binary_cross_entropy(pred, y.to(pred.dtype))
        measures['bce'] = bce.item()

        #numpy conversion for sklearn functions
        threshold = 0.5
        pred = (pred >= threshold).astype(int)
        pred = pred.numpy()
        y = y.numpy()

        accuracy = accuracy_score(y, pred)
        measures['accuracy'] = accuracy

        precision = precision_score(y, pred, zero_division=0)
        measures['precision'] = precision

        recall = recall_score(y, pred)
        measures['recall'] = recall

        f1 = f1_score(y, pred)
        measures['f1'] = f1

        cm = confusion_matrix(y, pred)
        measures['confusion_matrix'] = cm

    elif isinstance(ANFISmodel.last_layer, nn.Softmax):
        #pytorch calculation for ce calculation
        ce = nn.functional.cross_entropy(pred, y)
        measures['ce'] = ce.item()

        pred = ANFISmodel.classes_prediction(x)

        #numpy conversion for sklearn functions
        pred = pred.numpy()
        y = y.numpy()

        accuracy = accuracy_score(y, pred)
        measures['accuracy'] = accuracy

        precision = precision_score(y, pred, average='weighted', zero_division=0)
        measures['precision'] = precision

        recall = recall_score(y, pred, average='weighted')
        measures['recall'] = recall

        f1 = f1_score(y, pred, average='weighted')
        measures['f1'] = f1

        cm = confusion_matrix(y, pred, labels=list(range(ANFISmodel.consequents.shape[0])))
        measures['confusion_matrix'] = cm

    return measures

# OLS

In [19]:
class OLS():
    def __init__(self, epochs, gamma=0.01, loss_function=nn.functional.mse_loss, optimizer=None, optim_params=None, validation=0, early_stopping=None):
        #Hyperparameters
        self.epochs = epochs
        self.gamma = gamma

        #Premises Update
        self.loss_function = loss_function
        self.optimizer = optimizer
        self.optim_params = optim_params

        #History
        self.history = {'loss': torch.tensor([])}
        self.val_history = {'loss': torch.tensor([])}

        #EarlyStopping
        self.validation = validation
        self.early_stopping = early_stopping


    def __call__(self, ANFISmodel, loader, freezed=None):
        train_loader, val_loader = self.val_partition(loader)

        _ = self.obtain_metrics(ANFISmodel, train_loader, val_loader)
        ep = 0
        while (ep < self.epochs):

            self.consequentsUpdate(ANFISmodel, train_loader, freezed)
            self.premisesUpdate(ANFISmodel, train_loader, freezed)

            loss = self.obtain_metrics(ANFISmodel, train_loader, val_loader)
            if self.early_stopping != None:
                self.early_stopping(loss, ANFISmodel)
                if self.early_stopping.early_stop:
                    break;

            ep += 1
        _ = self.obtain_metrics(ANFISmodel, train_loader, val_loader)


    def consequentsUpdate(self, ANFISmodel, loader, freezed=None):
        x_train = loader.dataset.tensors[0]
        y_train = loader.dataset.tensors[1].clone().detach()
        if isinstance(ANFISmodel.last_layer, nn.Softmax):
            y_train = torch.eye(ANFISmodel.consequents.shape[0])[y_train].t()
        y_train = y_train.to(x_train.dtype)

        current_consequents = ANFISmodel.consequents
        new_consequents = torch.zeros_like(ANFISmodel.consequents)

        _, w_norm, _ = ANFISmodel.intermediate_values(x_train)
        xe = torch.cat((x_train, torch.ones(x_train.shape[0], 1)), dim=1)

        if freezed == None:
            freezed = torch.zeros(ANFISmodel.rules)

        fs = w_norm.unsqueeze(2).repeat(1, 1, xe.shape[1]).view(w_norm.shape[0], -1)
        X = xe.repeat(1, ANFISmodel.rules)

        for i in range(current_consequents.size(0)):
            flat_consequents, _, _, _ = torch.linalg.lstsq(fs * X, y_train[i])
            new_consequents[i] = torch.reshape(flat_consequents, (ANFISmodel.rules, xe.shape[1]))

        current_consequents[:, freezed == 0, :] = new_consequents[:, freezed == 0, :]

        ANFISmodel.set_consequents(current_consequents)


    def premisesUpdate(self, ANFISmodel, loader, freezed=None):
        if freezed == None:
            freezed = torch.zeros(ANFISmodel.rules)

        if (self.optimizer != None):
            optim = self.optimizer([ANFISmodel.fuzzify_layer.premises])
            if self.optim_params != None:
                self.apply_optimizer_parameters(optim)
            current_premises = ANFISmodel.premises.clone()

            for batch_x, batch_y in loader:
                batch_y_copy = batch_y.clone().detach()
                if not isinstance(ANFISmodel.last_layer, nn.Softmax):
                    batch_y_copy = batch_y_copy.to(batch_x.dtype)

                optim.zero_grad()
                pred = ANFISmodel(batch_x)
                loss = self.loss_function(pred, batch_y_copy)
                loss.backward()
                optim.step()

            new_premises = ANFISmodel.premises.clone()
            current_premises[:,freezed==0,:] = new_premises[:,freezed==0,:]

            ANFISmodel.set_premises(current_premises)

        else:
            new_premises = ANFISmodel.premises

            for batch_x, batch_y in loader:
                batch_y_copy = batch_y.clone().detach()
                if not isinstance(ANFISmodel.last_layer, nn.Softmax):
                    batch_y_copy = batch_y_copy.to(batch_x.dtype)
                if ANFISmodel.fuzzify_layer.premises.grad != None:
                    ANFISmodel.fuzzify_layer.premises.grad.data = torch.zeros_like(ANFISmodel.fuzzify_layer.premises.grad.data)
                    ANFISmodel.consequent_layer.consequents.grad.data = torch.zeros_like(ANFISmodel.consequent_layer.consequents.grad.data)
                pred = ANFISmodel(batch_x)
                loss = self.loss_function(pred, batch_y_copy)
                loss.backward()

                alpha = self.gamma / torch.sqrt(torch.sum(torch.pow(ANFISmodel.fuzzify_layer.premises.grad, 2)))

                vs = new_premises[:,:,0].t()
                sigmas = new_premises[:,:,1].t()

                new_vs = torch.zeros_like(vs)
                new_sigmas = torch.zeros_like(sigmas)

                _, w_norm, outputs_by_rule = ANFISmodel.intermediate_values(batch_x)

                if isinstance(ANFISmodel.last_layer, nn.Softmax):
                    batch_y_copy = torch.eye(ANFISmodel.consequents.shape[0])[batch_y_copy]

                for k in range(ANFISmodel.rules):
                    if freezed[k] == 0:
                        A = 4*alpha*(1/torch.pow(sigmas[k], 2))*(batch_x - vs[k])
                        B = 4*alpha*(1/torch.pow(sigmas[k], 3))*torch.pow((batch_x - vs[k]), 2)
                        wk = w_norm[:,k].unsqueeze(0).t()
                        fk = outputs_by_rule[:,:,k].t()

                        if pred.ndim == 1:
                            zk = ((fk-pred)*(batch_y_copy-pred)).unsqueeze(0).t()
                        else:
                            #
                            # SUM? MEAN? PROD?
                            #
                            zk = torch.sum((fk-pred)*(batch_y_copy-pred), dim=1).unsqueeze(0).t()
                        new_vs[k] = torch.sum(A*wk*zk, dim=0)
                        new_sigmas[k] = torch.sum(B*wk*zk, dim=0)

                new_premises[:, :, 0] += new_vs.t()
                new_premises[:, :, 1] += new_sigmas.t()

            ANFISmodel.set_premises(new_premises)

    def apply_optimizer_parameters(self, optimizer):
        for param_group in optimizer.param_groups:
            for key, new_value in self.optim_params.items():
                if key in param_group:
                    param_group[key] = new_value

    def val_partition(self, loader):
        if self.validation != 0:
            x_train, y_train = loader.dataset.tensors
            indices = torch.randperm(x_train.size(0))
            x_train = x_train[indices]
            y_train = y_train[indices]

            split_index = int(x_train.shape[0] * self.validation)

            x_val = x_train[:split_index]
            y_val = y_train[:split_index]
            x_train = x_train[split_index:]
            y_train = y_train[split_index:]

            train_loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size=loader.batch_size, shuffle=True)
            val_loader = data.DataLoader(data.TensorDataset(x_val, y_val), batch_size=loader.batch_size, shuffle=False)
        else:
            train_loader = loader
            val_loader = None

        return train_loader, val_loader

    def obtain_metrics(self, ANFISmodel, train_loader, val_loader):
        #Validation set
        if (val_loader != None):
            x_val = val_loader.dataset.tensors[0]
            y_val = val_loader.dataset.tensors[1]

            with torch.no_grad():
                pred = ANFISmodel(x_val)

            if isinstance(ANFISmodel.last_layer, nn.Softmax):
                val_loss = self.loss_function(pred, y_val)
            else:
                val_loss = self.loss_function(pred, y_val.to(pred.dtype))
                
            self.val_history['loss'] = torch.cat((self.val_history['loss'], torch.tensor([val_loss])))

            measures = obtain_measures(ANFISmodel, x_val, y_val)

            for measure in measures:
                if measure not in self.val_history:
                    self.val_history[measure] = torch.tensor([])
                self.val_history[measure] =  torch.cat((self.val_history[measure], torch.tensor([measures[measure]])))

        #Training set
        x_train = train_loader.dataset.tensors[0]
        y_train = train_loader.dataset.tensors[1]

        with torch.no_grad():
            pred = ANFISmodel(x_train)

        if isinstance(ANFISmodel.last_layer, nn.Softmax):
            loss = self.loss_function(pred, y_train)
        else:
            loss = self.loss_function(pred, y_train.to(pred.dtype))
        self.history['loss'] = torch.cat((self.history['loss'], torch.tensor([loss])))

        measures = obtain_measures(ANFISmodel, x_train, y_train)

        for measure in measures:
            if measure not in self.history:
                self.history[measure] = torch.tensor([])
            self.history[measure] =  torch.cat((self.history[measure], torch.tensor([measures[measure]])))

        if val_loader != None:
            loss = val_loss
        return loss

## Testing

### Multiclass

In [24]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [ ]:
X.info()

### Binary

#### Data

In [20]:
parkinsons = fetch_ucirepo(id=174)

X = parkinsons.data.features
y = parkinsons.data.targets

In [21]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 22 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   MDVP:Fo       195 non-null    float64
 1   MDVP:Fhi      195 non-null    float64
 2   MDVP:Flo      195 non-null    float64
 3   MDVP:Jitter   195 non-null    float64
 4   MDVP:Jitter   195 non-null    float64
 5   MDVP:RAP      195 non-null    float64
 6   MDVP:PPQ      195 non-null    float64
 7   Jitter:DDP    195 non-null    float64
 8   MDVP:Shimmer  195 non-null    float64
 9   MDVP:Shimmer  195 non-null    float64
 10  Shimmer:APQ3  195 non-null    float64
 11  Shimmer:APQ5  195 non-null    float64
 12  MDVP:APQ      195 non-null    float64
 13  Shimmer:DDA   195 non-null    float64
 14  NHR           195 non-null    float64
 15  HNR           195 non-null    float64
 16  RPDE          195 non-null    float64
 17  DFA           195 non-null    float64
 18  spread1       195 non-null    

In [22]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [23]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [24]:
x_test = torch.tensor(x_test, dtype=torch.float32)
x_train = torch.tensor(x_train, dtype=torch.float32)

In [25]:
y_train = torch.tensor(np.array(y_train), dtype=torch.float32).squeeze()
y_test = torch.tensor(np.array(y_test), dtype=torch.float32).squeeze()

In [26]:
x_train.shape

torch.Size([136, 22])

In [27]:
y_train.shape

torch.Size([136])

In [28]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 32, shuffle = True)
x_trainset = loader.dataset.tensors[0]
y_trainset = loader.dataset.tensors[1]

In [29]:
x_trainset.shape

torch.Size([136, 22])

In [30]:
y_trainset.shape

torch.Size([136])

#### ANFIS

In [31]:
model = rANFIS(x_train, init_fuzzy_rules=10, mf=gaussian2, output_type='binary')

In [32]:
model(x_trainset)

tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.8602, 0.5000, 0.5000,
         0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.8779, 0.5000, 0.5000, 0.5000,
         0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.8937,
         0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000,
         0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000,
         0.5000, 0.2741, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.6776, 0.5000,
         0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000,
         0.5000, 0.5000, 0.5000, 0.1644, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000,
         0.5000, 0.5000, 0.5000, 0.5000, 0.8812, 0.5000, 0.5000, 0.5000, 0.5000,
         0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.8238, 0.5000, 0.5000, 0.5000,
         0.8063, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000,
         0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000,
         0.5000, 0.5000, 0.5

In [33]:
y_trainset

tensor([1., 1., 0., 1., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 0., 1., 1.,
        1., 0., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1.,
        1., 0., 1., 1., 1., 1., 0., 0., 0., 1., 1., 0., 1., 0., 1., 1., 1., 1.,
        0., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 0., 1., 1., 0., 1., 1., 0.,
        0., 1., 1., 1., 1., 0., 1., 1., 1., 1., 0., 1., 1., 1., 1., 0., 1., 1.,
        1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 1., 0., 0., 1., 1., 1., 0., 0.,
        1., 1., 0., 1., 0., 1., 0., 1., 1., 0.])

#### OLS

In [97]:
epochs = 20
validation = 0.2
loss_func = nn.functional.binary_cross_entropy
optimizer = torch.optim.AdamW
params = {'lr': 0.01, 'weight_decay': 0.01}
#ols_eraly_stop = EarlyStopping(patience=10)


ols = OLS(epochs=epochs, validation=validation, loss_function=loss_func, optimizer=optimizer, optim_params=params)

#### Execution

In [98]:
ols(model, loader)

ValueError: Using a target size (torch.Size([27])) that is different to the input size (torch.Size([1, 27])) is deprecated. Please ensure they have the same size.